In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import RFE
import joblib


In [5]:
df = pd.read_csv("TrainDataset2025_Preprocessed_Iter.csv")

X = df.drop(columns=["ID", "RelapseFreeSurvival (outcome)"])
y = df["RelapseFreeSurvival (outcome)"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [6]:
# Coarse search ranges (log scale)
C_range_coarse = np.logspace(-2, 2.7, 10)      # 0.01 → 500
gamma_range_coarse = np.logspace(-4, 0, 10)    # 1e−4 → 1

# Empty results list
global_results = []

svr_linear = SVR(kernel="linear")


In [7]:
for k in range(15, 51):
    print(f"\n\n============================================")
    print(f"🔎 Nested Optimization for k = {k} features")
    print("============================================")

    # Step 1 — RFE
    rfe = RFE(estimator=svr_linear, n_features_to_select=k)
    X_rfe = rfe.fit_transform(X_scaled, y)

    # Step 2 — Coarse sweep
    best_coarse_mae = float("inf")
    best_coarse_C = None
    best_coarse_gamma = None

    for C in C_range_coarse:
        for gamma in gamma_range_coarse:
            svr = SVR(kernel="rbf", C=C, gamma=gamma)
            mae = -cross_val_score(
                svr, X_rfe, y,
                scoring="neg_mean_absolute_error", cv=5
            ).mean()

            if mae < best_coarse_mae:
                best_coarse_mae = mae
                best_coarse_C = C
                best_coarse_gamma = gamma

    print(f"Coarse best: C={best_coarse_C:.4f}, gamma={best_coarse_gamma:.6f}, MAE={best_coarse_mae:.4f}")

    # Step 3 — Fine search around coarse winners
    C_range_fine = np.linspace(best_coarse_C * 0.5, best_coarse_C * 1.5, 10)
    gamma_range_fine = np.linspace(best_coarse_gamma * 0.5, best_coarse_gamma * 1.5, 10)

    best_fine_mae = float("inf")
    best_fine_C = None
    best_fine_gamma = None

    for C2 in C_range_fine:
        for g2 in gamma_range_fine:
            svr = SVR(kernel="rbf", C=C2, gamma=g2)
            mae2 = -cross_val_score(
                svr, X_rfe, y,
                scoring="neg_mean_absolute_error", cv=5
            ).mean()

            if mae2 < best_fine_mae:
                best_fine_mae = mae2
                best_fine_C = C2
                best_fine_gamma = g2

    print(f"Fine best: C={best_fine_C:.4f}, gamma={best_fine_gamma:.6f}, MAE={best_fine_mae:.4f}")

    # Store global results
    global_results.append({
        "k": k,
        "MAE": best_fine_mae,
        "C": best_fine_C,
        "gamma": best_fine_gamma
    })




🔎 Nested Optimization for k = 15 features
Coarse best: C=501.1872, gamma=0.002154, MAE=22.2816
Fine best: C=473.3435, gamma=0.002274, MAE=22.2731


🔎 Nested Optimization for k = 16 features
Coarse best: C=501.1872, gamma=0.002154, MAE=22.2705
Fine best: C=473.3435, gamma=0.002274, MAE=22.2698


🔎 Nested Optimization for k = 17 features
Coarse best: C=501.1872, gamma=0.002154, MAE=22.3833
Fine best: C=751.7809, gamma=0.001317, MAE=22.2600


🔎 Nested Optimization for k = 18 features
Coarse best: C=501.1872, gamma=0.002154, MAE=22.3949
Fine best: C=751.7809, gamma=0.001077, MAE=22.2439


🔎 Nested Optimization for k = 19 features
Coarse best: C=501.1872, gamma=0.000774, MAE=22.3143
Fine best: C=751.7809, gamma=0.001161, MAE=22.2309


🔎 Nested Optimization for k = 20 features
Coarse best: C=501.1872, gamma=0.002154, MAE=22.0114
Fine best: C=250.5936, gamma=0.003232, MAE=22.0003


🔎 Nested Optimization for k = 21 features
Coarse best: C=150.5836, gamma=0.005995, MAE=22.1402
Fine best: C=17

In [8]:
nested_df = pd.DataFrame(global_results).sort_values("MAE")
nested_df


,k,MAE,C,gamma
5,20,22.000314,250.593617,0.003232
7,22,22.051504,125.486363,0.003664
8,23,22.056956,225.875453,0.005662
9,24,22.072832,225.875453,0.001556
10,25,22.084840,225.875453,0.005662
6,21,22.096857,175.680908,0.003664
11,26,22.135162,225.875453,0.001317
14,29,22.182257,584.718439,0.000903
12,27,22.206811,640.405910,0.000559
4,19,22.230924,751.780850,0.001161


In [9]:
best_model = nested_df.iloc[0]
best_model


k         20.000000
MAE       22.000314
C        250.593617
gamma      0.003232
Name: 5, dtype: float64

In [11]:
final_k = int(best_model["k"])
final_C = float(best_model["C"])
final_gamma = float(best_model["gamma"])

print(final_k, final_C, final_gamma)

# Refit RFE
rfe_final = RFE(estimator=svr_linear, n_features_to_select=final_k)
X_rfe_final = rfe_final.fit_transform(X_scaled, y)

# Train final SVR
svr_final = SVR(kernel="rbf", C=final_C, gamma=final_gamma)
svr_final.fit(X_rfe_final, y)

# Save artifacts
joblib.dump(rfe_final, "rfe_selector_nested_best.pkl")
joblib.dump(scaler, "svm_scaler_nested_best.pkl")
joblib.dump(svr_final, "svr_model_nested_best.pkl")

print("Saved nested optimized RFE + SVR model!")


20 250.59361681363623 0.003231652035047823
Saved nested optimized RFE + SVR model!
